# 0. Prepare SCOPe40 dataset

**Paper:** Dataset construction (SCOPe 2.08, identity ≤ 40%, 13,920 domains).

Download Foldseek `10-941cd33` and MMseqs2 `18-8cc5c`, classify official pdbstyle entries, build ground-truth Foldseek / MMseqs databases, and write `work/labels/scop_lookup.tsv`.

| | Path |
|--|------|
| Input | none (downloads Foldseek, MMseqs, SCOPe40 pdbstyle + classification) |
| Output | `bin/foldseek`, `bin/mmseqs` |
| | `work/GT_fasta/DB_aa.fasta`, `DB_di.fasta` |
| | `work/DB/foldseek_DB/`, `work/DB/mmseqs_DB/` |
| | `work/labels/scop_lookup.tsv` |
| Optional | `work/scope40_work_bundle.tar.gz` |

**Node:** login node is enough (`THREADS = 16`). ~30–60 min depending on download.

**Next:** copy predicted FASTA into `work/aa2di_fasta/` and `work/di2aa_fasta/`, then `1_build_predicted_dbs.ipynb`.

```bash
conda env create -f environment.yml   # or: conda activate ESM3_3Di_5090
conda activate ESM3_3Di_5090
```

Re-run from scratch: `rm -rf tmp work/DB work/GT_fasta work/labels bin` (keep `aa2di_fasta` / `di2aa_fasta`).


## Environment


In [ ]:
NOTEBOOK_NAME = "0_prepare_scope40.ipynb"

import os
import platform
import subprocess
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if not (cwd / NOTEBOOK_NAME).is_file():
    raise SystemExit(
        f"Start this notebook from the project root (cwd must contain {NOTEBOOK_NAME}). "
        f"Current cwd: {cwd}"
    )

CONDA_ENV = "ESM3_3Di_5090"
print("notebook:", NOTEBOOK_NAME)
print("cwd:", cwd)
print("python:", sys.executable)
print("version:", sys.version.split()[0])
print("platform:", platform.platform())
print("CONDA_DEFAULT_ENV:", os.environ.get("CONDA_DEFAULT_ENV", "(unset)"))

if CONDA_ENV not in sys.executable:
    expected = Path.home() / ".conda" / "envs" / CONDA_ENV / "bin" / "python"
    raise SystemExit(
        f"Kernel is not {CONDA_ENV} (current: {sys.executable}). "
        f"Select kernel {CONDA_ENV} and Restart. Do not pip into miniforge3 python3.12. "
        f"Expected: {expected}"
    )


def _bin_version(name: str) -> str:
    path = cwd / "bin" / name
    if not path.is_file():
        return "(not installed yet; run 0_prepare_scope40.ipynb)"
    try:
        proc = subprocess.run([str(path), "version"], capture_output=True, text=True, check=False)
        lines = (proc.stdout or proc.stderr or "").strip().splitlines()
        return lines[0] if lines else "(unknown)"
    except OSError as exc:
        return f"(failed: {exc})"


print("foldseek:", _bin_version("foldseek"))
print("mmseqs:", _bin_version("mmseqs"))


## Configuration

本格在五本 notebook 中**字节级相同**。改方法表、搜索参数或 URL 时：只改 `0_prepare_scope40.ipynb` 这一格，再整格复制到另外四本。发布前可用 checksum 核对五本是否一致。


In [ ]:
# =============================================================================
# Configuration — copy this entire cell into all five notebooks.
# Change methods or search parameters here in 0_prepare_scope40.ipynb, then
# paste the same cell into 1_build / 2a / 2b / 3_figures.
# =============================================================================
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
HOME = ROOT.parent

CONDA_ENV = "ESM3_3Di_5090"
FOLDSEEK_VERSION = "10-941cd33"
MMSEQS_VERSION = "18-8cc5c"

TEMP = ROOT / "tmp"
WORK_DIR = ROOT / "work"
BIN_DIR = ROOT / "bin"
WORK_TMP_DIR = WORK_DIR / "tmp"

GT_FASTA_DIR = WORK_DIR / "GT_fasta"
AA_FASTA = GT_FASTA_DIR / "DB_aa.fasta"
GT_DI_FASTA = GT_FASTA_DIR / "DB_di.fasta"

AA2DI_FASTA_DIR = WORK_DIR / "aa2di_fasta"
DI2AA_FASTA_DIR = WORK_DIR / "di2aa_fasta"

DBS_DIR = WORK_DIR / "DB"
FOLDSEEK_GT_DIR = DBS_DIR / "foldseek_DB"
MMSEQS_GT_DIR = DBS_DIR / "mmseqs_DB"

LABEL_DIR = WORK_DIR / "labels"
SCOP_LOOKUP = LABEL_DIR / "scop_lookup.tsv"
LEGACY_LABEL_DIR = WORK_DIR / "lable"

ALN_DIR = WORK_DIR / "aln"
METRICS_DIR = WORK_DIR / "metrics"
FIGURES_DIR = WORK_DIR / "figures"
TRANSLATION_METRICS_DIR = METRICS_DIR / "translation"
WORK_BUNDLE = WORK_DIR / "scope40_work_bundle.tar.gz"

FOLDSEEK_BIN = BIN_DIR / "foldseek"
MMSEQS_BIN = BIN_DIR / "mmseqs"

FOLDSEEK_URL = (
    "https://github.com/steineggerlab/foldseek/releases/download/"
    f"{FOLDSEEK_VERSION}/foldseek-linux-avx2.tar.gz"
)
MMSEQS_URL = (
    "https://github.com/soedinglab/MMseqs2/releases/download/"
    f"{MMSEQS_VERSION}/mmseqs-linux-avx2.tar.gz"
)
FOLDSEEK_TMP_DIR = TEMP / "foldseek"
MMSEQS_TMP_DIR = TEMP / "mmseqs"
FOLDSEEK_TARBALL = TEMP / "foldseek-linux-avx2.tar.gz"
MMSEQS_TARBALL = TEMP / "mmseqs-linux-avx2.tar.gz"

SCOP_CLA_NAME = "dir.cla.scope.2.08-stable.txt"
SCOP_DES_NAME = "dir.des.scope.2.08-stable.txt"
SOURCE_ARCHIVE_NAME = "pdbstyle-sel-gs-bib-40-2.08.tgz"
SCOP_CLA_FALLBACK = HOME / "SCOPE" / SCOP_CLA_NAME
HF_BASE = "https://huggingface.co/datasets/caijihuize/scope40_pdbstyle/resolve/main"

# Foldseek / predicted 3Di: aligned with new_scope40 easy-search
EASY_SEARCH_PARAMS = {
    "sensitivity": 9.5,
    "max_seqs": 2000,
    "evalue": 10.0,
    "threads": 64,
}
# MMseqs2: aligned with foldseek-analysis/scopbenchmark/scripts/runMMseqs.sh
MMSEQS_SEARCH_PARAMS = {
    "sensitivity": 7.5,
    "max_seqs": 2000,
    "evalue": 10000,
    "threads": 64,
    "add_backtrace": True,
}
PREPARE_THREADS = 16
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# Homology search methods. protocol must not be mixed as one AUC.
METHODS: list[dict] = [
    {"name": "Foldseek (AA+3Di)", "key": "foldseek", "engine": "foldseek", "aa2di": None, "protocol": "hitlist"},
    {"name": "MMseqs2", "key": "mmseqs", "engine": "mmseqs", "aa2di": None, "protocol": "catalog"},
    {"name": "ESM3-3Di", "key": "ESM3", "engine": "foldseek", "aa2di": "DB_ESM3_aa2di.fasta", "protocol": "hitlist"},
    {"name": "ESM3-LoRA", "key": "ESM3_LoRA", "engine": "foldseek", "aa2di": "DB_ESM3_LoRA_aa2di.fasta", "protocol": "hitlist"},
    {"name": "ProstT5 (translate)", "key": "ProstT5", "engine": "foldseek", "aa2di": "DB_ProstT5_translate_aa2di.fasta", "protocol": "hitlist"},
    {"name": "SaProt", "key": "SaProt", "engine": "foldseek", "aa2di": "DB_SaProt_aa2di.fasta", "protocol": "hitlist"},
]
# Translation accuracy (bidirectional). Homology search uses aa2di only.
TRANSLATION_METHODS: list[dict] = [
    {"name": "ESM3-3Di", "key": "ESM3", "aa2di": "DB_ESM3_aa2di.fasta", "di2aa": "DB_ESM3_di2aa.fasta"},
    {"name": "ESM3-LoRA", "key": "ESM3_LoRA", "aa2di": "DB_ESM3_LoRA_aa2di.fasta", "di2aa": "DB_ESM3_LoRA_di2aa.fasta"},
    {"name": "ProstT5 (translate)", "key": "ProstT5", "aa2di": "DB_ProstT5_translate_aa2di.fasta", "di2aa": "DB_ProstT5_translate_di2aa.fasta"},
    {"name": "SaProt", "key": "SaProt", "aa2di": "DB_SaProt_aa2di.fasta", "di2aa": "DB_SaProt_di2aa.fasta"},
]
PALETTE = {
    "Foldseek (AA+3Di)": "#2b5c8f",
    "MMseqs2": "#666666",
    "ESM3-3Di": "#d95f02",
    "ESM3-LoRA": "#1b9e77",
    "ProstT5 (translate)": "#7570b3",
    "SaProt": "#e7298a",
}


def run_cmd(argv: list[str]) -> None:
    """Print then run an external command. Logs are supplementary material."""
    print("[CMD]", " ".join(str(x) for x in argv), flush=True)
    subprocess.run([str(x) for x in argv], check=True)


def require_file(path: Path, hint: str) -> Path:
    """Fail with a pointer to the upstream notebook if a required file is missing."""
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}\n{hint}")
    return path


def meta_path(output: Path) -> Path:
    return output.with_name(output.name + ".meta.json")


def skip_if_exists(output: Path, payload: dict | None = None, skip_existing: bool = True) -> bool:
    """Skip when output exists. If payload is given, require a matching .meta.json.

    MMseqs TSV without a fingerprint is treated as stale (catalog protocol change).
    Other outputs without a fingerprint are kept; set SKIP_EXISTING=False to force.
    """
    if not skip_existing:
        return False
    if not output.is_file() or output.stat().st_size == 0:
        return False
    if payload is None:
        return True
    meta = meta_path(output)
    if not meta.is_file():
        if payload.get("engine") == "mmseqs":
            print(f"[rerun] {output.name}: no fingerprint; MMseqs catalog params need a fresh search")
            return False
        print(f"[warn] {output.name}: no fingerprint; keeping existing file (set SKIP_EXISTING=False to re-run)")
        return True
    try:
        stored = json.loads(meta.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return False
    if stored != payload:
        print(f"[rerun] {output.name}: Configuration changed")
        return False
    return True


def write_meta(output: Path, payload: dict) -> None:
    meta_path(output).write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def search_params_payload(engine: str, threads: int | None = None) -> dict:
    params = dict(MMSEQS_SEARCH_PARAMS if engine == "mmseqs" else EASY_SEARCH_PARAMS)
    params["engine"] = engine
    if threads is not None:
        params["threads"] = int(threads)
    return params


def method_by_key(method_key: str) -> dict:
    for row in METHODS:
        if row["key"] == method_key:
            return row
    raise KeyError(f"Unknown method_key: {method_key}")


def predicted_methods() -> list[dict]:
    return [row for row in METHODS if row["aa2di"] is not None]


def db_prefix(method_key: str) -> Path:
    return DBS_DIR / f"{method_key}_DB" / "DB"


def aln_tsv(method_key: str) -> Path:
    return ALN_DIR / f"{method_key}_easy.tsv"


def aln_tmp_dir(method_key: str) -> Path:
    return WORK_TMP_DIR / f"easy_{method_key}"


def metric_prefix(method_key: str) -> Path:
    return METRICS_DIR / f"{method_key}_easy"


def translation_per_seq_path(task: str, method_key: str) -> Path:
    return TRANSLATION_METRICS_DIR / f"{task}_{method_key}_per_seq.tsv"


def translation_summary_path(task: str) -> Path:
    return TRANSLATION_METRICS_DIR / f"{task}_summary.csv"


def scop_cla_path() -> Path:
    for path in (TEMP / SCOP_CLA_NAME, SCOP_CLA_FALLBACK):
        if path.is_file():
            return path
    return TEMP / SCOP_CLA_NAME


def work_ready() -> bool:
    return (
        (FOLDSEEK_GT_DIR / "DB").is_file()
        and (MMSEQS_GT_DIR / "DB").is_file()
        and AA_FASTA.is_file()
        and GT_DI_FASTA.is_file()
        and SCOP_LOOKUP.is_file()
    )


def ensure_work_dirs() -> None:
    for directory in (
        TEMP, BIN_DIR, GT_FASTA_DIR, AA2DI_FASTA_DIR, DI2AA_FASTA_DIR, DBS_DIR,
        LABEL_DIR, ALN_DIR, METRICS_DIR, TRANSLATION_METRICS_DIR, FIGURES_DIR, WORK_TMP_DIR,
    ):
        directory.mkdir(parents=True, exist_ok=True)
    legacy = LEGACY_LABEL_DIR / "scop_lookup.tsv"
    if not SCOP_LOOKUP.is_file() and legacy.is_file():
        shutil.copy2(legacy, SCOP_LOOKUP)
        print(f"[ok] migrated {legacy} -> {SCOP_LOOKUP}")


def cleanup_tmp(*, also_work_tmp: bool = True) -> None:
    """Remove tmp/ and work/tmp/ only. Keep work/ products and bin/."""
    targets = [TEMP]
    if also_work_tmp:
        targets.append(WORK_TMP_DIR)
    for path in targets:
        if path.exists():
            shutil.rmtree(path)
            print(f"[ok] cleaned {path}")
        else:
            print(f"[skip] {path} (absent)")


print("ROOT:", ROOT)
print("conda env:", CONDA_ENV)
print("Foldseek:", FOLDSEEK_VERSION, "MMseqs:", MMSEQS_VERSION)
print("METHODS:", [(m["key"], m["engine"], m["protocol"]) for m in METHODS])


## Run flags


In [ ]:
SKIP_EXISTING = True
THREADS = PREPARE_THREADS
MAKE_BUNDLE = True

TMP_WORK = TEMP / "work"
SOURCE_ARCHIVE = TEMP / SOURCE_ARCHIVE_NAME
SCOP_CLA = TEMP / SCOP_CLA_NAME
SCOP_DES = TEMP / SCOP_DES_NAME
EXTRACTED = TEMP / "pdbstyle-2.08"
SINGLE = TMP_WORK / "SCOPe40"
MULTI = TMP_WORK / "SCOPe40-multi"
OTHER = TMP_WORK / "SCOPe40-other"
BUILD_DB = TMP_WORK / "FoldseekDB"
BUILD_MMSEQS_DB = TMP_WORK / "MMseqsDB"
CLASSIFICATION = TMP_WORK / "structure_classification.tsv"

ensure_work_dirs()
for path in (TEMP, TMP_WORK, BIN_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("TEMP:", TEMP)
print("work/:", WORK_DIR)
print(f"SKIP_EXISTING={SKIP_EXISTING}  THREADS={THREADS}  MAKE_BUNDLE={MAKE_BUNDLE}")


## Helpers


In [ ]:
import csv
import tarfile

def count_models(path: Path) -> int:
    """Count MODEL records in a pdbstyle .ent file."""
    with path.open(encoding="utf-8", errors="ignore") as handle:
        return sum(line.startswith("MODEL") for line in handle)


def has_atoms(path: Path) -> bool:
    """True if the structure has ATOM or HETATM records."""
    with path.open(encoding="utf-8", errors="ignore") as handle:
        for line in handle:
            if line.startswith(("ATOM", "HETATM")):
                return True
    return False


def load_scop_classes() -> dict[str, str]:
    """Parse dir.cla: domain id -> SCOPe class (e.g. a.1.1.1)."""
    require_file(SCOP_CLA, hint="Run Step 2 (download SCOPe40 sources) first.")
    id_to_class: dict[str, str] = {}
    with SCOP_CLA.open(encoding="utf-8") as handle:
        for line in handle:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.rstrip().split("\t")
            if len(parts) >= 4:
                id_to_class[parts[0]] = parts[3]
    return id_to_class


def resolve_scop_class(seq_id: str, id_to_class: dict[str, str]) -> str:
    """Map Foldseek FASTA id to SCOPe class; fall back to parent if chain-split."""
    if seq_id in id_to_class:
        return id_to_class[seq_id]
    if "_" in seq_id:
        parent = seq_id.rsplit("_", 1)[0]
        if parent in id_to_class:
            return id_to_class[parent]
    return ""


def download_if_missing(url: str, dest: Path) -> Path:
    """wget/curl url -> dest unless SKIP_EXISTING and dest is non-empty."""
    if SKIP_EXISTING and dest.is_file() and dest.stat().st_size > 0:
        print(f"[skip] {dest.name}")
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    wget = shutil.which("wget")
    curl = shutil.which("curl")
    if wget:
        run_cmd([wget, "-O", str(dest), url])
    elif curl:
        run_cmd([curl, "-L", "-o", str(dest), url])
    else:
        raise RuntimeError("Need wget or curl to download")
    return dest


def install_release_binary(url: str, tarball: Path, extract_dir: Path, src_rel: str, dest: Path) -> Path:
    """Download an official AVX2 tarball and copy the binary into bin/."""
    if SKIP_EXISTING and dest.is_file() and os.access(dest, os.X_OK):
        print(f"[skip] {dest}")
        return dest
    download_if_missing(url, tarball)
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    run_cmd(["tar", "xzf", str(tarball), "-C", str(TEMP)])
    src = TEMP / src_rel
    if not src.is_file():
        raise FileNotFoundError(f"Extracted archive is missing {src}")
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dest)
    dest.chmod(dest.stat().st_mode | 0o111)
    print(f"[ok] {dest}")
    return dest


def extract_source(skip_existing: bool = True) -> Path:
    """Extract pdbstyle-sel-gs-bib-40-2.08.tgz under tmp/pdbstyle-2.08."""
    if skip_existing and EXTRACTED.is_dir() and any(EXTRACTED.rglob("*.ent")):
        print(f"[skip] already extracted: {EXTRACTED}")
        return EXTRACTED
    require_file(SOURCE_ARCHIVE, hint="Run Step 2 first.")
    with tarfile.open(SOURCE_ARCHIVE, "r:gz") as tf:
        root = EXTRACTED.parent.resolve()
        for member in tf.getmembers():
            destination = (EXTRACTED.parent / member.name).resolve()
            if root not in destination.parents and destination != root:
                raise RuntimeError(f"Unsafe archive path: {member.name}")
        tf.extractall(path=EXTRACTED.parent)
    if not EXTRACTED.is_dir():
        raise FileNotFoundError(f"Extract did not create {EXTRACTED}")
    return EXTRACTED


print("helpers ready")


## Step 1 — Install Foldseek and MMseqs2

Official Linux AVX2 builds. Binaries go to `bin/` (gitignored). If `/proc/cpuinfo` has no avx2, the binaries may not run.


In [ ]:
try:
    cpuinfo = Path("/proc/cpuinfo").read_text(encoding="utf-8", errors="ignore")
    if "avx2" not in cpuinfo.lower():
        print("[warn] /proc/cpuinfo has no avx2; linux-avx2 binaries may not run")
except OSError:
    pass

install_release_binary(
    FOLDSEEK_URL, FOLDSEEK_TARBALL, FOLDSEEK_TMP_DIR, "foldseek/bin/foldseek", FOLDSEEK_BIN,
)
install_release_binary(
    MMSEQS_URL, MMSEQS_TARBALL, MMSEQS_TMP_DIR, "mmseqs/bin/mmseqs", MMSEQS_BIN,
)
run_cmd([str(FOLDSEEK_BIN), "version"])
run_cmd([str(MMSEQS_BIN), "version"])


## Step 2 — Download SCOPe40 pdbstyle and classification

Hugging Face mirror of the official SCOPe 2.08 40% identity pdbstyle set (`caijihuize/scope40_pdbstyle`). Official Berkeley URLs (fallback):

- `https://scop.berkeley.edu/downloads/pdbstyle/pdbstyle-sel-gs-bib-40-2.08.tgz`
- `https://scop.berkeley.edu/downloads/parse/dir.cla.scope.2.08-stable.txt`


In [ ]:
download_if_missing(f"{HF_BASE}/{SOURCE_ARCHIVE_NAME}", SOURCE_ARCHIVE)
download_if_missing(f"{HF_BASE}/{SCOP_CLA_NAME}", SCOP_CLA)
download_if_missing(f"{HF_BASE}/{SCOP_DES_NAME}", SCOP_DES)
extract_source(skip_existing=SKIP_EXISTING)
n_ent = sum(1 for _ in EXTRACTED.rglob("*.ent"))
print(f"[ok] {SOURCE_ARCHIVE.name}  {SOURCE_ARCHIVE.stat().st_size / 2**20:.1f} MiB")
print(f"[ok] extracted {EXTRACTED}  .ent files={n_ent:,}")


## Step 3 — Classify pdbstyle entries

Single-model domains with atoms and a SCOPe class go to `SCOPe40`; multi-model to `SCOPe40-multi`; the rest to `SCOPe40-other`. Foldseek `createdb` uses the single-model set only.


In [ ]:
def classify_structures(skip_existing: bool = True) -> Path:
    if (
        skip_existing
        and CLASSIFICATION.is_file()
        and SINGLE.is_dir()
        and MULTI.is_dir()
        and OTHER.is_dir()
    ):
        print(f"[skip] classification exists: {CLASSIFICATION}")
        return CLASSIFICATION

    id_to_class = load_scop_classes()
    for directory in (SINGLE, MULTI, OTHER):
        if directory.exists():
            shutil.rmtree(directory)
        directory.mkdir(parents=True)

    pdb_files = sorted(EXTRACTED.rglob("*.ent"))
    if not pdb_files:
        raise FileNotFoundError(f"No .ent under {EXTRACTED}")

    rows: list[tuple[str, int, str, int, str]] = []
    for index, source in enumerate(pdb_files, start=1):
        models = count_models(source)
        domain_id = source.stem
        scop_class = id_to_class.get(domain_id, "")
        in_cla = int(bool(scop_class))
        if not has_atoms(source) or not in_cla:
            group, destination = "other", OTHER
        else:
            group = "multi" if models > 1 else "single"
            destination = MULTI if group == "multi" else SINGLE
        target = destination / source.name
        try:
            os.link(source, target)
        except OSError:
            shutil.copy2(source, target)
        rows.append((source.name, models, group, in_cla, scop_class))
        if index % 1000 == 0:
            print(f"classify {index:,}/{len(pdb_files):,}", flush=True)

    TMP_WORK.mkdir(parents=True, exist_ok=True)
    with CLASSIFICATION.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle, delimiter="\t")
        writer.writerow(["file", "model_count", "group", "in_cla", "scop_class"])
        writer.writerows(rows)
    print(
        f"[ok] single={sum(r[2] == 'single' for r in rows):,}, "
        f"multi={sum(r[2] == 'multi' for r in rows):,}, "
        f"other={sum(r[2] == 'other' for r in rows):,}"
    )
    return CLASSIFICATION


manifest = classify_structures(skip_existing=SKIP_EXISTING)
print("single:", SINGLE)
print("manifest:", manifest)


## Step 4 — Build Foldseek ground-truth DB

`createdb` on the single-model directory, then `lndb` for 3Di headers and `convert2fasta` for AA / 3Di FASTA.


In [ ]:
def build_foldseek_db(skip_existing: bool = True, threads: int = 16) -> Path:
    marker = BUILD_DB / "DB"
    if skip_existing and marker.is_file():
        print(f"[skip] Foldseek DB exists: {marker}")
        return marker
    require_file(FOLDSEEK_BIN, hint="Run Step 1 first.")
    if BUILD_DB.exists():
        shutil.rmtree(BUILD_DB)
    BUILD_DB.mkdir(parents=True)
    db = str(marker)
    run_cmd([str(FOLDSEEK_BIN), "createdb", str(SINGLE), db, "--threads", str(threads)])
    run_cmd([str(FOLDSEEK_BIN), "lndb", f"{db}_h", f"{db}_ss_h"])
    run_cmd([str(FOLDSEEK_BIN), "convert2fasta", db, f"{db}_aa.fasta"])
    run_cmd([str(FOLDSEEK_BIN), "convert2fasta", f"{db}_ss", f"{db}_di.fasta"])
    return marker


db = build_foldseek_db(skip_existing=SKIP_EXISTING, threads=THREADS)
print("foldseek db:", db)


## Step 5 — Build MMseqs2 ground-truth DB

MMseqs `createdb` on the AA FASTA from Step 4 (sequence baseline, no 3Di).


In [ ]:
def build_mmseqs_db(skip_existing: bool = True, threads: int = 16) -> Path:
    marker = BUILD_MMSEQS_DB / "DB"
    aa_fasta = BUILD_DB / "DB_aa.fasta"
    if skip_existing and marker.is_file():
        print(f"[skip] MMseqs DB exists: {marker}")
        return marker
    require_file(MMSEQS_BIN, hint="Run Step 1 first.")
    require_file(aa_fasta, hint="Run Step 4 first.")
    if BUILD_MMSEQS_DB.exists():
        shutil.rmtree(BUILD_MMSEQS_DB)
    BUILD_MMSEQS_DB.mkdir(parents=True)
    run_cmd([str(MMSEQS_BIN), "createdb", str(aa_fasta), str(marker), "--threads", str(threads)])
    return marker


mmseqs_db = build_mmseqs_db(skip_existing=SKIP_EXISTING, threads=THREADS)
print("mmseqs db:", mmseqs_db)


## Step 6 — Install products into `work/`

Copy DBs and FASTA into the layout used by later notebooks. Write `work/labels/scop_lookup.tsv` (domain id → SCOPe class) from dir.cla.


In [ ]:
def build_scop_lookup(aa_fasta: Path, output: Path) -> Path:
    id_to_class = load_scop_classes()
    ids = [
        line[1:].split()[0]
        for line in aa_fasta.open(encoding="utf-8")
        if line.startswith(">")
    ]
    matched = 0
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w", encoding="utf-8") as handle:
        for seq_id in ids:
            scop_class = resolve_scop_class(seq_id, id_to_class)
            if scop_class:
                handle.write(f"{seq_id}\t{scop_class}\n")
                matched += 1
    print(f"[ok] scop lookup: {output}  matched={matched:,} unmatched={len(ids) - matched:,}")
    return output


def _copy_db_tree(src: Path, dst: Path) -> None:
    if dst.exists():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            shutil.rmtree(dst)
    dst.mkdir(parents=True)
    for source in src.iterdir():
        if source.name in {"DB_aa.fasta", "DB_di.fasta"}:
            continue
        target = dst / source.name
        if source.is_symlink():
            shutil.copy2(source.resolve(), target)
        elif source.is_file():
            shutil.copy2(source, target)


def install_to_work(skip_existing: bool = True) -> Path:
    if skip_existing and (FOLDSEEK_GT_DIR / "DB").is_file() and AA_FASTA.is_file() and SCOP_LOOKUP.is_file():
        print(f"[skip] work/ already installed: {WORK_DIR}")
        return WORK_DIR
    require_file(BUILD_DB / "DB", hint="Run Step 4 first.")
    require_file(BUILD_MMSEQS_DB / "DB", hint="Run Step 5 first.")
    ensure_work_dirs()
    _copy_db_tree(BUILD_DB, FOLDSEEK_GT_DIR)
    _copy_db_tree(BUILD_MMSEQS_DB, MMSEQS_GT_DIR)
    for name in ("DB_aa.fasta", "DB_di.fasta"):
        src = BUILD_DB / name
        require_file(src, hint="Run Step 4 first.")
        shutil.copy2(src, GT_FASTA_DIR / name)
    LABEL_DIR.mkdir(parents=True, exist_ok=True)
    for stale in LABEL_DIR.iterdir():
        if stale.is_file() and stale.name != SCOP_LOOKUP.name:
            stale.unlink()
    lookup = build_scop_lookup(AA_FASTA, SCOP_LOOKUP)
    print(f"[ok] installed {WORK_DIR}")
    print(f"   GT_fasta: {AA_FASTA.name}, {GT_DI_FASTA.name}")
    print(f"   DB: {FOLDSEEK_GT_DIR.name}, {MMSEQS_GT_DIR.name}")
    print(f"   labels: {lookup}")
    return WORK_DIR


install_to_work(skip_existing=SKIP_EXISTING)


## Step 7 — Optional work bundle

Tarball of `GT_fasta` + GT DBs + `labels` for sharing. Skip if `MAKE_BUNDLE = False`.


In [ ]:
def make_work_bundle(skip_existing: bool = True) -> Path | None:
    if not MAKE_BUNDLE:
        print("[skip] MAKE_BUNDLE=False")
        return None
    if skip_existing and WORK_BUNDLE.is_file() and WORK_BUNDLE.stat().st_size > 0:
        print(f"[skip] bundle exists: {WORK_BUNDLE}")
        return WORK_BUNDLE
    require_file(FOLDSEEK_GT_DIR / "DB", hint="Run Step 6 first.")
    require_file(SCOP_LOOKUP, hint="Run Step 6 first.")
    with tarfile.open(WORK_BUNDLE, "w:gz") as tf:
        for rel in ("GT_fasta", "DB/foldseek_DB", "DB/mmseqs_DB", "labels"):
            path = WORK_DIR / rel
            tf.add(path, arcname=f"scope40_work/{rel}")
    print(f"[ok] bundle: {WORK_BUNDLE} ({WORK_BUNDLE.stat().st_size / 2**20:.1f} MiB)")
    return WORK_BUNDLE


bundle = make_work_bundle(skip_existing=SKIP_EXISTING)
print("bundle:", bundle)


## Verify


In [ ]:
checks = {
    "foldseek": FOLDSEEK_BIN,
    "mmseqs": MMSEQS_BIN,
    "foldseek DB": FOLDSEEK_GT_DIR / "DB",
    "mmseqs DB": MMSEQS_GT_DIR / "DB",
    "DB_aa.fasta": AA_FASTA,
    "DB_di.fasta": GT_DI_FASTA,
    "scop_lookup": SCOP_LOOKUP,
}
missing = []
for name, path in checks.items():
    ok = path.is_file()
    print(("[ok]" if ok else "[missing]"), f"{name}: {path}")
    if not ok:
        missing.append(name)
if missing:
    raise SystemExit(f"Prepare incomplete, missing: {missing}")
n_lookup = sum(1 for _ in SCOP_LOOKUP.open())
n_aa = sum(1 for line in AA_FASTA.open() if line.startswith(">"))
print(f"verify OK — domains in FASTA={n_aa:,}  scop_lookup rows={n_lookup:,}")
print("Next: copy predicted FASTA, then 1_build_predicted_dbs.ipynb")


## Cleanup


In [ ]:
cleanup_tmp(also_work_tmp=True)
print("Cleanup done. Products remain under work/ and bin/.")
